In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMCON10", con=connection2)

connection2.close()




In [2]:
base2

,oricenasicod,cenasicod,concod,condes,conubi,conestregcod,contipamb,conact,consecamb,conobs,conextflg,conextdir,conextcoordx,conextcoordy,conlocalcod
0,1,265,C-23,,,0,None,None,NaN,None,0,None,None,None,None
1,1,265,C13A,,,0,None,None,NaN,None,0,None,None,None,None
2,1,265,C-21A,NUTRICION,,0,None,None,NaN,None,0,None,None,None,None
3,1,265,C-33A,TRAUMATOLOGIA,,0,None,None,NaN,None,0,None,None,None,None
4,1,265,C-13A,CIRUGIA,,0,None,None,NaN,None,0,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42586,1,010,TE109,ODONTOLOGIA,VIRTUAL,0,1,01,109.0,CONSULTORIO VIRTUAL,0,,None,None,None
42587,1,431,119,TELEMEDICINA 1,CENT. TELEM.,1,1,01,119.0,Atención medica,0,,None,None,None
42588,1,008,ENF14,S. OCUPACIONAL,REMOTO,0,1,02,56.0,.,0,,None,None,None
42589,1,396,PSIC4,PSICOLOGIA,PSICOLOGIA,1,1,04,999.0,PADOMI,0,,None,None,None


In [3]:
a=base2[base2.duplicated(subset=["concod","cenasicod"])]
a

,oricenasicod,cenasicod,concod,condes,conubi,conestregcod,contipamb,conact,consecamb,conobs,conextflg,conextdir,conextcoordx,conextcoordy,conlocalcod


In [4]:
base2.columns

Index(['oricenasicod', 'cenasicod', 'concod', 'condes', 'conubi',
       'conestregcod', 'contipamb', 'conact', 'consecamb', 'conobs',
       'conextflg', 'conextdir', 'conextcoordx', 'conextcoordy',
       'conlocalcod'],
      dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmcon10 ()')
base2.to_sql(name='tmp_cmcon10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmcon10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_cmcon10 
ALTER COLUMN	cenasicod	TYPE	CHARACTER (3),
ALTER COLUMN	concod	TYPE	CHARACTER (5),
ALTER COLUMN	condes	TYPE	CHARACTER (15),
ALTER COLUMN	conubi	TYPE	CHARACTER (15);


UPDATE cmcon10 
SET 


oricenasicod                = t.oricenasicod, 
cenasicod                   = t.cenasicod, 
concod                      = t.concod, 
condes                      = t.condes, 
conubi                      = t.conubi, 
conestregcod                = t.conestregcod, 
contipamb                   = t.contipamb, 
conact                      = t.conact, 
consecamb                   = t.consecamb, 
conobs                      = t.conobs, 
conextflg                   = t.conextflg, 
conextdir                   = t.conextdir, 
conextcoordx                = t.conextcoordx, 
conextcoordy                = t.conextcoordy, 
conlocalcod                 = t.conlocalcod



FROM tmp_cmcon10 t 
WHERE cmcon10.concod = t.concod and cmcon10.cenasicod = t.cenasicod and cmcon10.cenasicod IS NOT NULL and t.cenasicod IS NOT NULL ;

INSERT INTO cmcon10 
(oricenasicod, cenasicod, concod, condes, conubi, conestregcod, contipamb, conact, consecamb, conobs, conextflg, conextdir, conextcoordx, conextcoordy, conlocalcod) 
SELECT 
oricenasicod, cenasicod, concod, condes, conubi, conestregcod, contipamb, conact, consecamb, conobs, conextflg, conextdir, conextcoordx, conextcoordy, conlocalcod

FROM tmp_cmcon10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmcon10 
    WHERE cmcon10.concod = tmp_cmcon10.concod and cmcon10.cenasicod = tmp_cmcon10.cenasicod and cmcon10.cenasicod IS NOT NULL and tmp_cmcon10.cenasicod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmcon10;
DELETE FROM cmcon10 WHERE cenasicod ='';
"""


c2= text(query2)
connection3.execute(c2)
base2 = pd.read_sql_query(f"SELECT * FROM cmcon10", con=connection3)

connection3.close()


In [6]:
base2.columns

Index(['oricenasicod', 'cenasicod', 'concod', 'condes', 'conubi',
       'conestregcod', 'contipamb', 'conact', 'consecamb', 'conobs',
       'conextflg', 'conextdir', 'conextcoordx', 'conextcoordy',
       'conlocalcod'],
      dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmcon10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""

ALTER TABLE tmp_cmcon10 
ALTER COLUMN	cenasicod	TYPE	CHARACTER (3),
ALTER COLUMN	concod	TYPE	CHARACTER (5),
ALTER COLUMN	condes	TYPE	CHARACTER (15),
ALTER COLUMN	conubi	TYPE	CHARACTER (15);


UPDATE dim_consult 
SET 

des_con       = t.condes,
ubi_con       = t.conubi

FROM tmp_cmcon10 t 
WHERE dim_consult.cod_cas = t.cenasicod AND dim_consult.cod_con=t.concod AND dim_consult.cod_cas  IS NOT NULL;


INSERT INTO dim_consult (cod_cas, cod_con, des_con, ubi_con) 
SELECT cenasicod, concod, condes, conubi      

FROM tmp_cmcon10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_consult 
    WHERE dim_consult.cod_cas = tmp_cmcon10.cenasicod and dim_consult.cod_con = tmp_cmcon10.concod
);

DROP TABLE tmp_cmcon10;

"""



c1= text(query)
connection4.execute(c1)
connection4.close()